In [2]:
!pip install face-alignment scikit-image pandas

  Using cached face_alignment-1.4.1-py2.py3-none-any.whl.metadata (7.4 kB)
  Using cached torch-2.7.0-cp39-cp39-win_amd64.whl.metadata (29 kB)
  Using cached numba-0.60.0-cp39-cp39-win_amd64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.43.0-cp39-cp39-win_amd64.whl.metadata (4.9 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached face_alignment-1.4.1-py2.py3-none-any.whl (30 kB)
Using cached numba-0.60.0-cp39-cp39-win_amd64.whl (2.7 MB)
   ---------------------------------------- 0.0/212.4 MB ? eta -:--:--
   - -------------------------------------- 5.5/212.4 MB 33.6 MB/s eta 0:00:07
   -- ------------------------------------- 14.9/212.4 MB 42.7 MB/s eta 0:00:05
   --- ------------------------------------ 18.6/212.4 MB 45.2 MB/s eta 0:00:05
   --- -------------------

In [6]:
import os
import pandas as pd
import numpy as np
from skimage import io
from tqdm import tqdm
import face_alignment

# Force CPU to avoid CUDA errors
fa = face_alignment.FaceAlignment('2D', device='cpu', flip_input=False)


dataset_dir = 'data'
output_dir = 'output_csv'
os.makedirs(output_dir, exist_ok=True)

def extract_landmarks(image_path):
    try:
        input_img = io.imread(image_path)
        landmarks = fa.get_landmarks(input_img)
        if landmarks is None:
            return None
        return landmarks[0].flatten().tolist()  # 68 x (x, y) => 136 values
    except:
        return None

def process_folder(folder_path, label):
    data = []
    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)
        features = extract_landmarks(file_path)
        if features:
            data.append(features + [label])
    return data

def process_dataset(split):
    real_path = os.path.join(dataset_dir, split, 'real')
    fake_path = os.path.join(dataset_dir, split, 'fake')

    print(f"Processing {split} set...")
    real_data = process_folder(real_path, 0)
    fake_data = process_folder(fake_path, 1)

    all_data = real_data + fake_data
    col_names = [f'pt_{i}' for i in range(136)] + ['label']
    df = pd.DataFrame(all_data, columns=col_names)

    df.to_csv(os.path.join(output_dir, f"{split}_biomarkers.csv"), index=False)
    print(f"Saved {split} data to CSV.")

# Run on both train and test
process_dataset('train')
process_dataset('test')


Processing train set...


  0%|                                                                                        | 0/50000 [00:00<?, ?it/s]C:\Users\Dell\OneDrive\Desktop\Deepfake\venv\lib\site-packages\face_alignment\api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")
  1%|▋                                                                             | 423/50000 [00:07<13:17, 62.15it/s]C:\Users\Dell\OneDrive\Desktop\Deepfake\venv\lib\site-packages\face_alignment\api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")
100%|████████████████████████████████████████████████████████████████████████████| 50000/50000 [13:44<00:00, 60.63it/s]


Saved train data to CSV.
Processing test set...


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [02:43<00:00, 61.10it/s]

Saved test data to CSV.
